In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append(".")
print(" Ready")

 Ready


In [1]:
%%writefile agents/data_extraction_agent.py
import json
import re
from typing import Dict, List

from tools.llm_clients import reasoning_llm
from tools.logger import log_agent_event

AGENT_NAME = "data_extraction_agent"


def _call_llm(prompt: str) -> str:
    """Thin wrapper around the LLM call, kept separate so tests can mock a plain
    function instead of monkeypatching a Pydantic model's method (which Pydantic v2
    blocks for undeclared attributes)."""
    return reasoning_llm.invoke(prompt).content


def _build_extraction_prompt(sub_question: str, answer: str) -> str:
    return f"""Extract discrete, atomic factual claims from the text below, related to the
question: "{sub_question}"

Rules:
- Each fact must be a single, self-contained, checkable statement.
- Do NOT include opinions, vague generalities, or marketing language.
- If the text contains no clear factual claims (e.g. it's irrelevant, garbled, or empty),
  return an empty list.
- Respond with ONLY a JSON array of strings, nothing else. No markdown, no explanation.

Text:
\"\"\"{answer}\"\"\"

Example valid response: ["Fact one stated plainly.", "Fact two stated plainly."]
Example for irrelevant/no facts: []
"""


def _parse_json_facts(raw_text: str) -> List[str]:
    cleaned = raw_text.strip()
    cleaned = re.sub(r"^```(json)?", "", cleaned).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, list):
            return [str(f).strip() for f in parsed if str(f).strip()]
        return []
    except json.JSONDecodeError:
        return []


def extract_facts_from_result(search_result: Dict) -> List[Dict]:
    """Extracts atomic facts from one search result. Never raises: any failure
    (LLM error, unparseable output) returns an empty list, tagged in the log."""
    sub_question = search_result.get("sub_question", "")
    answer = search_result.get("answer", "")
    sources = search_result.get("sources", [])

    if not answer.strip():
        return []

    try:
        raw_output = _call_llm(_build_extraction_prompt(sub_question, answer))
        raw_facts = _parse_json_facts(raw_output)
    except Exception as e:
        log_agent_event(AGENT_NAME, "extraction_failed", sub_question=sub_question,
                        error=f"{type(e).__name__}: {e}")
        return []

    facts = [
        {"claim": fact_text, "sub_question": sub_question, "sources": sources}
        for fact_text in raw_facts
    ]

    log_agent_event(AGENT_NAME, "extraction_done", sub_question=sub_question,
                    facts_found=len(facts))
    return facts


def data_extraction_node(state: Dict) -> Dict:
    """LangGraph node: extracts facts from every search result in state."""
    search_results = state.get("search_results", [])
    if not search_results:
        return {"errors": [{"agent": AGENT_NAME, "error": "no search_results to extract from"}]}

    all_facts: List[Dict] = []
    errors: List[Dict] = []

    for result in search_results:
        try:
            facts = extract_facts_from_result(result)
            all_facts.extend(facts)
        except Exception as e:
            errors.append({
                "agent": AGENT_NAME,
                "sub_question": result.get("sub_question", "unknown"),
                "error": f"{type(e).__name__}: {e}",
            })

    return {
        "extracted_facts": all_facts,
        "errors": errors,
        "model_usage_log": [{"agent": AGENT_NAME, "provider": "groq", "calls": len(search_results)}],
    }

Overwriting agents/data_extraction_agent.py


In [3]:
from agents.web_search_agent import search_one
from agents.data_extraction_agent import extract_facts_from_result

search_result = search_one("What are the main safety risks of solid-state EV batteries?")
facts = extract_facts_from_result(search_result)

print(f"Extracted {len(facts)} facts:\n")
for f in facts:
    print("-", f["claim"])
    print("  sources:", [s["url"] for s in f["sources"][:1]])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Gemini unavailable (ClientError), falling back to Tavily...


2026-09-30 21:42:58,651 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3
2026-09-30 21:42:59,824 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=What are the main safety risks of solid-state EV batteries? facts_found=14


Extracted 14 facts:

- Continuous charging and discharging cycles lead to a gradual decline in battery capacity.
  sources: ['https://www.batterypowertips.com/what-are-the-main-challenges-in-developing-solid-state-batteries-for-evs']
- Frequent rapid charging accelerates the decline in battery capacity.
  sources: ['https://www.batterypowertips.com/what-are-the-main-challenges-in-developing-solid-state-batteries-for-evs']
- Liquid electrolytes introduce risks of leakage.
  sources: ['https://www.batterypowertips.com/what-are-the-main-challenges-in-developing-solid-state-batteries-for-evs']
- Liquid electrolytes introduce risks of dendrite formation.
  sources: ['https://www.batterypowertips.com/what-are-the-main-challenges-in-developing-solid-state-batteries-for-evs']
- Dendrite formation is characterized by needle-like structures.
  sources: ['https://www.batterypowertips.com/what-are-the-main-challenges-in-developing-solid-state-batteries-for-evs']
- Removing flammable liquid electro

In [4]:
garbage_result = search_one("asdfgh qwerty zxcv")
garbage_facts = extract_facts_from_result(garbage_result)

print(f"Extracted {len(garbage_facts)} facts from a nonsense query")
print("(0 is the correct, desired outcome here)")

2026-09-30 21:57:01,498 | INFO | research_system | [web_search_agent] search_done status=failed provider=none sources=0


Gemini unavailable (ClientError), falling back to Tavily...
Extracted 0 facts from a nonsense query
(0 is the correct, desired outcome here)


In [5]:
from agents.data_extraction_agent import data_extraction_node

state = {
    "search_results": [
        search_one("Tesla 4680 battery production status 2026"),
        search_one("How does lithium iron phosphate differ from NMC chemistry?"),
    ]
}

out = data_extraction_node(state)
print(f"Total facts extracted: {len(out['extracted_facts'])}")
print(f"Errors: {len(out['errors'])}")
for f in out["extracted_facts"][:5]:
    print("-", f["claim"])

Gemini unavailable (ClientError), falling back to Tavily...


2026-09-30 21:59:37,119 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3


Gemini unavailable (ClientError), falling back to Tavily...


2026-09-30 21:59:40,090 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3
2026-09-30 21:59:41,850 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=Tesla 4680 battery production status 2026 facts_found=5
2026-09-30 21:59:42,329 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=How does lithium iron phosphate differ from NMC chemistry? facts_found=8


Total facts extracted: 13
Errors: 0
- As of January 28, 2026, Tesla recommenced building Model Y battery packs with internally produced 4680 cells.
- As of January 2026, Tesla has not abandoned continued development of the 4680 cell.
- Elon Musk projected that 4680 production would ramp up significantly in 2026.
- The 2026 'Generation 3' 4680 cells have reached an energy density of 300 Wh/kg.
- The 2026 'Generation 3' 4680 cells allow for a 15% increase in range without increasing the physical size of the battery pack.


In [2]:
%%writefile tests/test_data_extraction_agent.py
import agents.data_extraction_agent as dea


def test_extract_facts_valid_json(monkeypatch):
    monkeypatch.setattr(dea, "_call_llm", lambda p: '["Fact A.", "Fact B."]')
    facts = dea.extract_facts_from_result({"sub_question": "q", "answer": "some text", "sources": []})
    assert len(facts) == 2
    assert facts[0]["claim"] == "Fact A."


def test_extract_facts_handles_markdown_fence(monkeypatch):
    monkeypatch.setattr(dea, "_call_llm", lambda p: '```json\n["Fact A."]\n```')
    facts = dea.extract_facts_from_result({"sub_question": "q", "answer": "some text", "sources": []})
    assert len(facts) == 1


def test_extract_facts_malformed_json_returns_empty(monkeypatch):
    monkeypatch.setattr(dea, "_call_llm", lambda p: "this is not json at all")
    facts = dea.extract_facts_from_result({"sub_question": "q", "answer": "some text", "sources": []})
    assert facts == []


def test_extract_facts_empty_answer_skips_llm_call(monkeypatch):
    called = {"value": False}
    def fake_call(p):
        called["value"] = True
        return "[]"
    monkeypatch.setattr(dea, "_call_llm", fake_call)
    facts = dea.extract_facts_from_result({"sub_question": "q", "answer": "", "sources": []})
    assert facts == [] and called["value"] is False


def test_extract_facts_llm_exception_returns_empty(monkeypatch):
    def boom(p):
        raise RuntimeError("API down")
    monkeypatch.setattr(dea, "_call_llm", boom)
    facts = dea.extract_facts_from_result({"sub_question": "q", "answer": "text", "sources": []})
    assert facts == []


def test_node_aggregates_across_results(monkeypatch):
    monkeypatch.setattr(dea, "_call_llm", lambda p: '["Fact X."]')
    state = {"search_results": [
        {"sub_question": "q1", "answer": "text1", "sources": []},
        {"sub_question": "q2", "answer": "text2", "sources": []},
    ]}
    out = dea.data_extraction_node(state)
    assert len(out["extracted_facts"]) == 2


def test_node_empty_search_results(monkeypatch):
    out = dea.data_extraction_node({"search_results": []})
    assert out["errors"] and "extracted_facts" not in out

Overwriting tests/test_data_extraction_agent.py


In [1]:
%%writefile tools/llm_clients.py
import os
import time
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from google import genai
from google.genai import types
from tavily import TavilyClient

load_dotenv()

GROQ_KEY = os.getenv("GROQ_API_KEY")
GEMINI_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_KEY = os.getenv("TAVILY_API_KEY")

assert GROQ_KEY, "Missing GROQ_API_KEY"
assert GEMINI_KEY, "Missing GEMINI_API_KEY"
assert TAVILY_KEY, "Missing TAVILY_API_KEY"

reasoning_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)
gemini_client = genai.Client(api_key=GEMINI_KEY)
tavily_client = TavilyClient(api_key=TAVILY_KEY)


def call_llm(prompt: str) -> str:
    """Shared wrapper around the reasoning LLM call. Every agent should import and
    use THIS function rather than calling reasoning_llm.invoke directly — it keeps
    the LangChain/Pydantic object out of agent code entirely, so tests can mock a
    plain function instead of monkeypatching a Pydantic model (which Pydantic v2
    blocks for undeclared attributes, as hit on Day 9)."""
    return reasoning_llm.invoke(prompt).content


def research_with_fallback(query: str) -> dict:
    """Tries Gemini (grounded search + citations) first. Falls back to Tavily
    automatically on any Gemini failure, so the pipeline never fully blocks
    on one provider's outage."""
    try:
        resp = gemini_client.models.generate_content(
            model="gemini-3.6-flash",
            contents=query,
            config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())])
        )
        sources = []
        if resp.candidates[0].grounding_metadata:
            sources = [
                {"title": c.web.title, "url": c.web.uri}
                for c in resp.candidates[0].grounding_metadata.grounding_chunks
            ]
        return {"answer": resp.text, "sources": sources, "provider": "gemini"}
    except Exception as e:
        print(f" Gemini unavailable ({type(e).__name__}), falling back to Tavily...")
        results = tavily_client.search(query=query, max_results=3)
        sources = [{"title": r["title"], "url": r["url"]} for r in results["results"]]
        answer = " ".join(r.get("content", "")[:300] for r in results["results"])
        return {"answer": answer, "sources": sources, "provider": "tavily"}

Overwriting tools/llm_clients.py


In [2]:
%%writefile agents/data_extraction_agent.py
import json
import re
from typing import Dict, List

from tools.llm_clients import call_llm
from tools.logger import log_agent_event

AGENT_NAME = "data_extraction_agent"


def _build_extraction_prompt(sub_question: str, answer: str) -> str:
    return f"""Extract discrete, atomic factual claims from the text below, related to the
question: "{sub_question}"

Rules:
- Each fact must be a single, self-contained, checkable statement.
- Do NOT include opinions, vague generalities, or marketing language.
- If the text contains no clear factual claims (e.g. it's irrelevant, garbled, or empty),
  return an empty list.
- Respond with ONLY a JSON array of strings, nothing else. No markdown, no explanation.

Text:
\"\"\"{answer}\"\"\"

Example valid response: ["Fact one stated plainly.", "Fact two stated plainly."]
Example for irrelevant/no facts: []
"""


def _parse_json_facts(raw_text: str) -> List[str]:
    cleaned = raw_text.strip()
    cleaned = re.sub(r"^```(json)?", "", cleaned).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, list):
            return [str(f).strip() for f in parsed if str(f).strip()]
        return []
    except json.JSONDecodeError:
        return []


def extract_facts_from_result(search_result: Dict) -> List[Dict]:
    """Extracts atomic facts from one search result. Never raises: any failure
    (LLM error, unparseable output) returns an empty list, tagged in the log."""
    sub_question = search_result.get("sub_question", "")
    answer = search_result.get("answer", "")
    sources = search_result.get("sources", [])

    if not answer.strip():
        return []

    try:
        raw_output = call_llm(_build_extraction_prompt(sub_question, answer))
        raw_facts = _parse_json_facts(raw_output)
    except Exception as e:
        log_agent_event(AGENT_NAME, "extraction_failed", sub_question=sub_question,
                        error=f"{type(e).__name__}: {e}")
        return []

    facts = [
        {"claim": fact_text, "sub_question": sub_question, "sources": sources}
        for fact_text in raw_facts
    ]

    log_agent_event(AGENT_NAME, "extraction_done", sub_question=sub_question,
                    facts_found=len(facts))
    return facts


def data_extraction_node(state: Dict) -> Dict:
    """LangGraph node: extracts facts from every search result in state."""
    search_results = state.get("search_results", [])
    if not search_results:
        return {"errors": [{"agent": AGENT_NAME, "error": "no search_results to extract from"}]}

    all_facts: List[Dict] = []
    errors: List[Dict] = []

    for result in search_results:
        try:
            facts = extract_facts_from_result(result)
            all_facts.extend(facts)
        except Exception as e:
            errors.append({
                "agent": AGENT_NAME,
                "sub_question": result.get("sub_question", "unknown"),
                "error": f"{type(e).__name__}: {e}",
            })

    return {
        "extracted_facts": all_facts,
        "errors": errors,
        "model_usage_log": [{"agent": AGENT_NAME, "provider": "groq", "calls": len(search_results)}],
    }

Overwriting agents/data_extraction_agent.py


In [ ]:
%%writefile tests/test_data_extraction_agent.py
import agents.data_extraction_agent as dea


def test_extract_facts_valid_json(monkeypatch):
    monkeypatch.setattr(dea, "call_llm", lambda p: '["Fact A.", "Fact B."]')
    facts = dea.extract_facts_from_result({"sub_question": "q", "answer": "some text", "sources": []})
    assert len(facts) == 2
    assert facts[0]["claim"] == "Fact A."


def test_extract_facts_handles_markdown_fence(monkeypatch):
    monkeypatch.setattr(dea, "call_llm", lambda p: '```json\n["Fact A."]\n```')
    facts = dea.extract_facts_from_result({"sub_question": "q", "answer": "some text", "sources": []})
    assert len(facts) == 1


def test_extract_facts_malformed_json_returns_empty(monkeypatch):
    monkeypatch.setattr(dea, "call_llm", lambda p: "this is not json at all")
    facts = dea.extract_facts_from_result({"sub_question": "q", "answer": "some text", "sources": []})
    assert facts == []


def test_extract_facts_empty_answer_skips_llm_call(monkeypatch):
    called = {"value": False}
    def fake_call(p):
        called["value"] = True
        return "[]"
    monkeypatch.setattr(dea, "call_llm", fake_call)
    facts = dea.extract_facts_from_result({"sub_question": "q", "answer": "", "sources": []})
    assert facts == [] and called["value"] is False


def test_extract_facts_llm_exception_returns_empty(monkeypatch):
    def boom(p):
        raise RuntimeError("API down")
    monkeypatch.setattr(dea, "call_llm", boom)
    facts = dea.extract_facts_from_result({"sub_question": "q", "answer": "text", "sources": []})
    assert facts == []


def test_node_aggregates_across_results(monkeypatch):
    monkeypatch.setattr(dea, "call_llm", lambda p: '["Fact X."]')
    state = {"search_results": [
        {"sub_question": "q1", "answer": "text1", "sources": []},
        {"sub_question": "q2", "answer": "text2", "sources": []},
    ]}
    out = dea.data_extraction_node(state)
    assert len(out["extracted_facts"]) == 2


def test_node_empty_search_results(monkeypatch):
    out = dea.data_extraction_node({"search_results": []})
    assert out["errors"] and "extracted_facts" not in out